# 🎬 AI-Powered Movie Recommendation Chatbot — Agno
**Applied AI Institute — Full Stack AI Capstone 2026**

## Architecture Overview

```
User Query
    │
    ▼
┌──────────────────────────────────────────────────────────────────┐
│                     Agno Agent  (single object)                  │
│                                                                  │
│   Agent(                                                         │
│     model   = OpenAIChat('gpt-4o'),                              │
│     tools   = [search_abt, search_taxonomy,                      │
│                search_oscars_rag, search_tmdb_by_id,             │
│                search_web],                                      │
│     storage = SqliteAgentStorage(),   ◀── multi-turn memory      │
│     instructions = [...],             ◀── agent persona          │
│     add_history_to_messages = True,   ◀── auto-injects history   │
│     show_tool_calls = True                                       │
│   )                                                              │
│                                                                  │
│   Tool calls resolved automatically by Agno run-loop            │
│   No explicit graph wiring required                              │
└──────────────────────────────────────────────────────────────────┘
    │
    ▼
 Personalised Recommendation Response
```

## Data Sources
| Source | File | Purpose |
|--------|------|---------|
| Agent Instructions | `use_case_Movie_Recommendation.xlsx` | System prompt + persona |
| ABT | `movie_abt_enriched_FINAL.xlsx` | 9,742 enriched movies |
| Taxonomy (TAG) | `Movie_Recommendation_TaxonomyV2.xlsx` | 24 semantic genre clusters |
| Oscars PDF (RAG) | `The_98th_Academy_Awards_2026.pdf` | 2026 Oscar nominations |
| TMDB API | Live call by movie ID | Full movie details fallback |
| Web Search | Serper | Latest news & streaming info |


## 1. Install Dependencies

In [1]:
!pip install agno

  Using cached agno-2.5.5-py3-none-any.whl.metadata (37 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached python_multipart-0.0.22-py3-none-any.whl.metadata (1.8 kB)
  Using cached rich-14.3.3-py3-none-any.whl.metadata (18 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.2-py3-none-any.whl.metadata (4.3 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached

In [2]:
!pip install cryptography jinja2 
!pip install requests
!pip install isodate  

  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
Using cached isodate-0.7.2-py3-none-any.whl (22 kB)


In [2]:
!pip uninstall -y langchain langchain-community  
!pip install langchain==1.2.10 langchain-community  

Found existing installation: langchain 1.2.10
Uninstalling langchain-1.2.10:
  Successfully uninstalled langchain-1.2.10
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
  Using cached langchain-1.2.10-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
Using cached langchain-1.2.10-py3-none-any.whl (111 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain]/2 [langchain]


In [11]:

## Install all required packages
!pip install openai 
!pip install langchain-openai 
!pip install pandas openpyxl faiss-cpu pypdf sentence-transformers
!pip install langchain langchain-community
!pip install python-dotenv sqlalchemy
!pip install pypdf



print('Dependencies ready. Uncom##!pip insrtall ;anglangchain-communityment above lines if running for the first time.')


Dependencies ready. Uncom##!pip insrtall ;anglangchain-communityment above lines if running for the first time.


## 2. Imports & Configuration

In [4]:
import os  
import ast  
import json  
import requests  
import pandas as pd  
from pathlib import Path  
from dotenv import load_dotenv  
  
# Agno  
from agno.agent import Agent  
from agno.models.openai import OpenAIChat  
# from agno.memory import SqliteMemory  # intentionally omitted  
from agno.tools import tool  
  
# LangChain for RAG (embeddings + FAISS)  
from langchain_openai import AzureOpenAIEmbeddings  
from langchain_community.vectorstores import FAISS  
from langchain_community.document_loaders import PyPDFLoader  

load_dotenv()
print('Imports complete.')


Imports complete.


## 3. User-Defined Data Folder & API Keys

In [8]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────────
DATA_FOLDER   = Path('../data/input')   # <-- change to your folder

USE_CASE_FILE = DATA_FOLDER / 'use_case_Movie_Recommendation_v2.xlsx'
ABT_FILE      = DATA_FOLDER / 'movie_abt_enriched_FINAL.xlsx'
TAXONOMY_FILE = DATA_FOLDER / 'Movie_Recommendation_TaxonomyV2.xlsx'
OSCARS_PDF    = DATA_FOLDER / 'The_98th_Academy_Awards_2026.pdf'

# SQLite file for Agno session memory
MEMORY_DB     = DATA_FOLDER / 'movie_agent_memory.db'

AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT=os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION=os.getenv("LLM_MODEL_VERSION") 
TMDB_API_KEY   = os.getenv('TMDB_API_KEY')
SERPER_API_KEY = os.getenv('SERPER_API_KEY')

LLM_MODEL        = os.getenv("LLM_MODEL")
#EMBED_MODEL      = 'text-embedding-3-small'
MAX_ABT_RESULTS  = 5
MAX_TAG_RESULTS  = 10
RAG_TOP_K        = 4
SESSION_ID       = 'movie-bot-session-001'  # change per user

print(f'Data folder: {DATA_FOLDER.resolve()}')
for f in [USE_CASE_FILE, ABT_FILE, TAXONOMY_FILE, OSCARS_PDF]:
    status = '✅' if f.exists() else '❌ NOT FOUND'
    print(f'  {status}  {f.name}')


Data folder: /Users/asathi/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/ai-powered-movie-recommendation/data/input
  ✅  use_case_Movie_Recommendation_v2.xlsx
  ✅  movie_abt_enriched_FINAL.xlsx
  ✅  Movie_Recommendation_TaxonomyV2.xlsx
  ✅  The_98th_Academy_Awards_2026.pdf


## 4. Load All Data Sources

In [20]:
# ─── 4a: Load agent instructions ────────────────────────────────────────────
def load_agent_instructions(filepath: Path) -> dict:
    df = pd.read_excel(filepath, sheet_name='Agent Instructions')
    df.columns = [c.strip() for c in df.columns]
    return {str(r['prompt_type']).strip(): str(r['prompt_value']).strip()
            for _, r in df.iterrows()}
instructions       = load_agent_instructions(USE_CASE_FILE)
AGENT_SYSTEM_PROMPT = instructions.get('agent_prompt', '')
print('System prompt loaded:')
print(AGENT_SYSTEM_PROMPT, '...')


System prompt loaded:
You are a friendly and knowledgeable movie expert. Your ONLY role is to help users discover and learn about movies. Do not answer questions unrelated to movies.

PHASE 1 — UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief, engaging conversation. Ask ONE leading question at a time across these dimensions (do not ask all at once):
  • Viewing context: Are you watching alone, with family, friends, or a date?
  • Audience: Who is the audience — adults, teens, young children, mixed group?
  • Genre mood: What genre or emotional mood are you in? (e.g., action, comedy, thriller, romance, drama, sci-fi, documentary)
  • Decade/era: Any preference for era or decade? (classic, 1980s–90s, modern, recent)
  • Country/language: Any preference for country of origin or spoken language?
  • Oscar/award interest: Are you interested in critically acclaimed or award-nominated films?
Stop asking when you have enough to m

In [21]:
# ─── 4b: Load ABT ────────────────────────────────────────────────────────────
def load_abt(filepath: Path) -> pd.DataFrame:
    df = pd.read_excel(filepath)
    df = df.rename(columns={'Unnamed: 0': 'row_id'})
    for col in ['movielens_genres', 'genres', 'spoken_languages', 'origin_country']:
        if col in df.columns:
            df[col] = df[col].astype(str)
    df['title_lower'] = df['title'].str.lower().str.strip()
    df['year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
    return df

abt_df = load_abt(ABT_FILE)
print(f'ABT loaded: {len(abt_df):,} movies')
print(abt_df[['title', 'year', 'vote_average', 'movielens_genres']].head(3).to_string())


ABT loaded: 9,742 movies
              title    year  vote_average                             movielens_genres
0         Toy Story  1995.0         7.970  Adventure|Animation|Children|Comedy|Fantasy
1           Jumanji  1995.0         7.244                   Adventure|Children|Fantasy
2  Grumpier Old Men  1995.0         6.482                               Comedy|Romance


In [22]:
# ─── 4c: Load Taxonomy ───────────────────────────────────────────────────────
def load_taxonomy(filepath: Path) -> pd.DataFrame:
    df = pd.read_excel(filepath)
    df = df.rename(columns={'Unnamed: 0': 'row_id'})
    def safe_parse(val):
        try:
            return ast.literal_eval(str(val)) if not isinstance(val, list) else val
        except Exception:
            return []
    df['included_movies'] = df['included_movies'].apply(safe_parse)
    return df

taxonomy_df = load_taxonomy(TAXONOMY_FILE)
print(f'Taxonomy loaded: {len(taxonomy_df)} genre clusters')
print('Genres:', list(taxonomy_df['custom_genre']))


Taxonomy loaded: 24 genre clusters
Genres: ['Clever Heist Capers', 'Global Espionage Thrillers', 'Psych-Crime Thrillers', 'Samurai Assassin Revenge', 'Galactic Saga Adventures', 'Viral Apocalypse Thrillers', 'Urban Grit & Velocity', 'Galactic Frontier Epics', 'Heist & Hostage Havoc', 'Emotional Southern & New York Dramas', 'Robin Hood Rebellion', 'Gotham Vigilante Saga', 'Whimsical Inventor Adventures', 'Twisted Justice Thrillers', 'Con & Crossroads', 'WWII Undersea & Survival Thrillers', 'Wartime Rebel Ops', 'Epic Middle-earth Quests', 'Nostalgic Coming-of-Age Comedy', 'Haunted Havens', 'Mutant Legacy Saga', 'Epic Kung Fu Legends', 'Wizarding Epic Saga', 'Marvel Heroic Saga']


In [25]:
pages    = PyPDFLoader(str(OSCARS_PDF)).load()
pages

[Document(metadata={'producer': 'Skia/PDF m144', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36', 'creationdate': '2026-02-03T18:59:53+00:00', 'title': 'The 98th Academy Awards | 2026', 'moddate': '2026-02-03T18:59:53+00:00', 'source': '../data/input/The_98th_Academy_Awards_2026.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='EXPERIENCE OVER NINE DECADES OF THE OSCARS FROM 1927 TO 2026\nTHE 98TH ACADEMY AWARDS | 2026\nDolby Theatre at Ovation Hollywood\nSunday, March 15, 2026\nHonoring movies released in 2025\nSHARE\nNOMINEES\n98th Oscars Nominations Announced by Actors Danielle Brooks & Lewis Pullm98th Oscars Nominations Announced by Actors Danielle Brooks & Lewis Pullm……\n1920s 1930s 1940s 1\n1965 1966 1967\n2026\nVISIT THE LARGEST MUSEUM IN THE WORLD DEDICATED TO THE ART OF MOVIEMAKING — THE ACADEMY MUSEUM\n2/3/26, 10:59 AM The 98th Academy Awards | 2026\nhttps://www.oscars.org/o

In [23]:
# ─── 4d: Build RAG index from Oscars PDF ─────────────────────────────────────
def build_rag_index(pdf_path: Path):
    print(f'Loading: {pdf_path.name}')
    pages    = PyPDFLoader(str(pdf_path)).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=600, chunk_overlap=150,
        separators=['\n\n', '\n', ' ', '']
    )
    docs       = splitter.split_documents(pages)
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL, openai_api_key=OPENAI_API_KEY)
    store      = FAISS.from_documents(docs, embeddings)
    print(f'RAG index built: {len(docs)} chunks from {len(pages)} pages.')
    return store

rag_store = build_rag_index(OSCARS_PDF)
# Optional persistence:
# rag_store.save_local(str(DATA_FOLDER / 'oscars_faiss_index'))
# rag_store = FAISS.load_local(str(DATA_FOLDER / 'oscars_faiss_index'), embeddings)


Loading: The_98th_Academy_Awards_2026.pdf


NameError: name 'RecursiveCharacterTextSplitter' is not defined

## 5. Define Tools

In Agno, tools are plain Python functions decorated with `@tool`.
The decorator extracts the docstring as the tool description for the LLM.
No separate schema class or JSON schema definition is needed.


In [ ]:
# ─── Tool 1: ABT Search ───────────────────────────────────────────────────────
@tool
def search_abt(query: str) -> str:
    """
    Search the Analytics-Based Table (ABT) of 9,742 enriched movies.
    Matches title, genre, language, decade, or plot keywords.
    Returns: title, year, genres, rating, runtime, tagline, overview,
    budget, revenue, vote_count, popularity, tmdbId, poster_path.
    Use as the PRIMARY search tool before falling back to TMDB API or web search.
    """
    q = query.lower()

    title_matches = abt_df[abt_df['title_lower'].str.contains(q, na=False)]
    genre_matches = abt_df[
        abt_df['movielens_genres'].str.lower().str.contains(q, na=False) |
        abt_df['genres'].str.lower().str.contains(q, na=False)
    ]
    text_matches  = abt_df[
        abt_df['overview'].str.lower().str.contains(q, na=False) |
        abt_df['tagline'].str.lower().str.contains(q, na=False)
    ]

    combined = pd.concat([title_matches, genre_matches, text_matches]).drop_duplicates('movieId')
    combined = combined.sort_values(['vote_average', 'popularity'], ascending=False)
    top = combined.head(MAX_ABT_RESULTS)

    if top.empty:
        return f'No ABT results for "{query}". Try search_taxonomy or search_web.'

    rows = []
    for _, r in top.iterrows():
        budget_str = f"\n  Budget: ${int(r['budget']):,}  |  Revenue: ${int(r['revenue']):,}" \
            if pd.notna(r.get('budget')) and r['budget'] > 0 else ''
        poster = f"\n  Poster: https://image.tmdb.org/t/p/w500{r['poster_path']}" \
            if pd.notna(r.get('poster_path')) else ''
        rows.append(
            f"Title: {r['title']} ({int(r['year']) if pd.notna(r.get('year')) else 'N/A'})"
            f"\n  Genres: {r['movielens_genres']}"
            f"\n  Rating: {r['vote_average']:.1f}/10 ({int(r['vote_count']) if pd.notna(r['vote_count']) else '?'} votes)"
            f"\n  Popularity: {r['popularity']:.1f}  |  Runtime: {int(r['runtime']) if pd.notna(r['runtime']) else '?'} min"
            f"\n  Language: {r['original_language']}  |  Country: {r['origin_country']}"
            f"\n  Tagline: {r['tagline']}"
            f"\n  Overview: {str(r['overview'])[:300]}..."
            f"{budget_str}"
            f"\n  TMDB ID: {int(r['tmdbId']) if pd.notna(r['tmdbId']) else 'N/A'}"
            f"{poster}"
        )
    return f'ABT results for "{query}":\n\n' + '\n\n---\n'.join(rows)

print('search_abt defined')


In [ ]:
# ─── Tool 2: Taxonomy / TAG Genre Search ─────────────────────────────────────
@tool
def search_taxonomy(genre_query: str) -> str:
    """
    Search the TAG (Taxonomy-Assisted Generation) genre clusters.
    Use for genre or mood queries: 'heist caper', 'spy thriller',
    'galactic adventure', 'coming-of-age comedy', 'kung fu', etc.
    Returns cluster name, description, and list of included movie titles.
    """
    q = genre_query.lower()
    taxonomy_df['_score'] = taxonomy_df.apply(lambda r: (
        (2 if q in str(r['custom_genre']).lower() else 0) +
        (1 if q in str(r['description']).lower() else 0) +
        (1 if q in str(r['exemplar_overview']).lower() else 0)
    ), axis=1)

    matches = taxonomy_df[taxonomy_df['_score'] > 0].sort_values('_score', ascending=False)
    if matches.empty:
        matches = taxonomy_df.head(3)
        prefix = f'No exact match for "{genre_query}". Closest clusters:\n\n'
    else:
        prefix = f'TAG genre matches for "{genre_query}":\n\n'

    results = []
    for _, r in matches.head(2).iterrows():
        results.append(
            f"Genre Cluster: {r['custom_genre']}"
            f"\n  Description: {r['description']}"
            f"\n  Example movies: {', '.join(r['included_movies'][:MAX_TAG_RESULTS])}"
        )
    return prefix + '\n\n---\n'.join(results)

print('search_taxonomy defined')


In [ ]:
# ─── Tool 3: RAG — Oscars PDF Search ─────────────────────────────────────────
@tool
def search_oscars_rag(query: str) -> str:
    """
    Search the 98th Academy Awards 2026 PDF for nomination and award information.
    Use for any Oscar-related questions: Best Picture nominees, acting nominations,
    directing nominees, original score, screenplay, visual effects, etc.
    Always cite this source when providing Oscar facts.
    """
    docs = rag_store.similarity_search(query, k=RAG_TOP_K)
    if not docs:
        return f'No Oscar 2026 information found for: "{query}"'
    chunks = [
        f'[Chunk {i+1} | Page {d.metadata.get("page", "?")}]:\n{d.page_content}'
        for i, d in enumerate(docs)
    ]
    return (
        f'Oscars 2026 (The_98th_Academy_Awards_2026.pdf) results for "{query}":\n\n'
        + '\n\n'.join(chunks)
    )

print('search_oscars_rag defined')


In [ ]:
# ─── Tool 4: TMDB API Exact Search by Movie ID ───────────────────────────────
@tool
def search_tmdb_by_id(tmdb_id: str) -> str:
    """
    Fetch complete movie details from the TMDB API using a numeric movie ID.
    Use as a FALLBACK when ABT, taxonomy, RAG, and web search lack sufficient detail.
    Returns: title, director, cast, genres, rating, runtime, tagline, overview,
    budget, revenue, homepage, poster_path.
    The TMDB ID is available in ABT search results under 'TMDB ID'.
    """
    url     = f'https://api.themoviedb.org/3/movie/{tmdb_id}'
    headers = {'Authorization': f'Bearer {TMDB_API_KEY}', 'accept': 'application/json'}
    r = requests.get(url, headers=headers, params={'append_to_response': 'credits'})

    if r.status_code != 200:
        return f'TMDB API error {r.status_code} for ID {tmdb_id}: {r.text[:200]}'

    d       = r.json()
    credits = d.get('credits', {})
    cast    = [c['name'] for c in credits.get('cast', [])[:5]]
    director = next((c['name'] for c in credits.get('crew', []) if c['job'] == 'Director'), 'N/A')
    genres   = [g['name'] for g in d.get('genres', [])]

    return (
        f"TMDB Details — ID {tmdb_id}:"
        f"\n  Title: {d.get('title')} ({str(d.get('release_date',''))[:4]})"
        f"\n  Director: {director}"
        f"\n  Cast: {', '.join(cast)}"
        f"\n  Genres: {', '.join(genres)}"
        f"\n  Rating: {d.get('vote_average')}/10 ({d.get('vote_count')} votes)"
        f"\n  Runtime: {d.get('runtime')} min"
        f"\n  Tagline: {d.get('tagline')}"
        f"\n  Overview: {str(d.get('overview',''))[:400]}"
        f"\n  Budget: ${d.get('budget',0):,}  |  Revenue: ${d.get('revenue',0):,}"
        f"\n  Homepage: {d.get('homepage')}"
        f"\n  Poster: https://image.tmdb.org/t/p/w500{d.get('poster_path')}"
    )

print('search_tmdb_by_id defined')


In [ ]:
# ─── Tool 5: Web Search (Serper) ─────────────────────────────────────────────
@tool
def search_web(query: str) -> str:
    """
    Search the web for current movie information.
    Use as the final fallback after ABT, taxonomy, RAG, and TMDB searches.
    Best for: streaming availability, recent reviews, box office results,
    upcoming releases, and any information not in the other data sources.
    """
    url     = 'https://google.serper.dev/search'
    headers = {'X-API-KEY': SERPER_API_KEY, 'Content-Type': 'application/json'}
    try:
        resp = requests.post(url, headers=headers,
                             json={'q': f'{query} movie', 'num': 5}, timeout=8)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f'Web search failed: {e}'

    items = []
    if 'answerBox' in data:
        ab = data['answerBox']
        items.append(f"Quick Answer: {ab.get('answer', ab.get('snippet', ''))}")
    for item in data.get('organic', [])[:4]:
        items.append(
            f"Title: {item.get('title')}"
            f"\n  Link: {item.get('link')}"
            f"\n  Snippet: {item.get('snippet')}"
        )
    return f'Web results for "{query}":\n\n' + '\n\n---\n'.join(items) if items else 'No results.'

print('search_web defined')


## 6. Build the Agno Agent

The entire agent is constructed in **one `Agent(...)` call**.
- `SqliteAgentStorage` gives persistent multi-turn memory per `session_id`
- `add_history_to_messages=True` automatically injects past conversation turns
- `show_tool_calls=True` prints tool calls for debugging
- No explicit graph, no state schema, no edge routing required


In [ ]:
# Build extended system instructions for the Agno agent
TOOL_GUIDE = """

Tool usage priority order:
1. search_taxonomy  — first choice for genre/mood/vibe queries (curated clusters)
2. search_abt       — primary source for movie details (9,742 enriched movies)
3. search_oscars_rag — mandatory for any Oscar, award, or nomination questions
4. search_tmdb_by_id — when TMDB ID is known and full cast/crew detail is needed
5. search_web       — last resort for streaming, news, release dates, reviews

When displaying a recommended movie, always include from the ABT where available:
- Tagline, runtime, original language, country of origin
- Vote average + vote count, popularity score
- Budget and revenue (if non-zero)
- Poster image URL
- TMDB ID (for deep-dive lookups)

Always begin each recommendation with one sentence explaining WHY you are
recommending this film based on what the user has told you in this conversation.
"""

agent = Agent(
    name        = 'MovieBot',
    model       = OpenAIChat(id=LLM_MODEL, api_key=OPENAI_API_KEY),
    tools       = [search_abt, search_taxonomy, search_oscars_rag,
                   search_tmdb_by_id, search_web],
    instructions= [AGENT_SYSTEM_PROMPT + TOOL_GUIDE],
    storage     = SqliteAgentStorage(
        table_name = 'movie_sessions',
        db_file    = str(MEMORY_DB)
    ),
    add_history_to_messages   = True,   # auto multi-turn memory
    num_history_responses     = 6,      # retain last 6 turns in context
    show_tool_calls           = True,   # debug: show which tools fire
    markdown                  = True,   # format responses as markdown
    temperature               = 0.3,
)

print('Agno Agent created:')
print(f'  Model:   {LLM_MODEL}')
print(f'  Tools:   {[t.__name__ for t in [search_abt, search_taxonomy, search_oscars_rag, search_tmdb_by_id, search_web]]}')
print(f'  Memory:  SQLite → {MEMORY_DB}')
print(f'  Session: {SESSION_ID}')


## 7. Conversation Runner

In [ ]:
def chat(user_input: str, session_id: str = SESSION_ID) -> str:
    """
    Run one conversation turn.
    Agno handles memory automatically via SqliteAgentStorage.
    Pass the same session_id across turns for multi-turn continuity.
    """
    response = agent.run(user_input, session_id=session_id)
    # Agno returns a RunResponse; extract text content
    if hasattr(response, 'content'):
        return response.content
    return str(response)


def run_demo_conversation(session_id: str = SESSION_ID):
    """Run the same 5-turn demo conversation as the LangGraph version."""
    turns = [
        "Hi! I'm looking for a great movie to watch tonight.",
        "I love clever heist movies and spy thrillers from the 1990s.",
        "Are any of those Oscar nominated? Specifically the 2026 Oscars?",
        "Tell me more about Sinners — what's its TMDB ID and full details?",
        "Where can I stream Sinners right now?"
    ]
    for turn in turns:
        print(f'\n{"="*70}')
        print(f'USER: {turn}')
        print('-'*70)
        response = chat(turn, session_id)
        print(f'AGENT: {response}')

print('chat() and run_demo_conversation() ready.')


## 8. Run the Demo Conversation

In [ ]:
# Runs exactly the same 5-turn scenario as the LangGraph notebook.
# NOTE: Requires valid OPENAI_API_KEY, TMDB_API_KEY, SERPER_API_KEY
run_demo_conversation()


## 9. Interactive Chat (REPL)

In [ ]:
def interactive_chat(session_id: str = SESSION_ID):
    print('🎬 Movie Recommendation Bot (Agno)')
    print('   Memory: SQLite — history persists across restarts')
    print('   Type your question. Enter "quit" to exit.\n')
    print(f'USER PROMPT: {instructions.get("user","")}')
    while True:
        user_input = input('\nYou: ').strip()
        if not user_input or user_input.lower() in ('quit', 'exit', 'q'):
            print('Goodbye!')
            break
        response = chat(user_input, session_id)
        print(f'\nBot: {response}')

# Uncomment to run:
# interactive_chat()
print('interactive_chat() ready — uncomment last line to run.')


## 10. Agno Architecture Notes & Framework Comparison

### Agno Agent Internals
```
agent.run(message, session_id)
  │
  ├─ Loads history from SqliteAgentStorage (last N turns)
  ├─ Builds prompt: system instructions + history + user message
  ├─ Calls LLM (GPT-4o) with tool schemas
  ├─ If tool_calls present → executes tools → re-invokes LLM
  │    (loop continues until no more tool calls)
  ├─ Saves turn to SQLite
  └─ Returns RunResponse
```

---

## LangGraph vs Agno — Full Comparison

### Architecture Philosophy
| Dimension | LangGraph | Agno |
|-----------|-----------|------|
| Mental model | Explicit **directed graph** (nodes + edges) | **Single agent object** with implicit loop |
| State | Typed `StateGraph` with `TypedDict` | Internal `RunResponse`; session in storage backend |
| Control flow | `add_conditional_edges()` — you define every branch | Agno's run-loop handles tool→LLM→tool automatically |
| Memory | Manual: pass state dict across calls; checkpointing optional | Built-in: `SqliteAgentStorage`, `PostgresStorage`, etc. |
| Multi-turn | Developer manages message accumulation via `add_messages` | `add_history_to_messages=True` handles it automatically |

### Implementation Similarities
| Aspect | Both Frameworks |
|--------|----------------|
| LLM backend | OpenAI GPT-4o via function-calling |
| Tool definition | Python functions with descriptive docstrings |
| Tool count | Same 5 tools: ABT, Taxonomy, RAG, TMDB, Web |
| RAG layer | FAISS + LangChain embeddings (identical code) |
| Data sources | Same 4 files + TMDB API + Serper |
| System prompt | Loaded from `use_case_Movie_Recommendation.xlsx` |

### Implementation Differences
| Aspect | LangGraph | Agno |
|--------|-----------|------|
| Lines of agent code | ~80 (graph + nodes + edges) | ~20 (single `Agent()` call) |
| Tool decorator | `@tool` from `langchain.tools` | `@tool` from `agno.tools` |
| Tool binding | `llm.bind_tools(tools)` then `ToolNode` | Passed directly as `tools=[...]` to Agent |
| Routing | `add_conditional_edges` with custom `route_after_agent` function | Automatic inside Agno run-loop |
| Memory | Custom `user_prefs` dict + `add_messages` annotated list | `SqliteAgentStorage` persisted automatically |
| Graph visualisation | `agent_graph.get_graph().draw_mermaid()` | Not applicable |
| Debugging | Node-by-node trace via LangSmith | `show_tool_calls=True` in Agent config |
| Streaming | `agent_graph.stream()` | `agent.run(..., stream=True)` |

### When to Choose Each
| Use Case | Recommended Framework |
|----------|-----------------------|
| Production at scale with audit trails | **LangGraph** — explicit graph = full observability |
| Rapid prototyping and iteration | **Agno** — minimal boilerplate, fast to deploy |
| Complex branching / multi-agent workflows | **LangGraph** — conditional edges give fine-grained control |
| Standard single-agent chatbot | **Agno** — built-in memory + tool loop covers most needs |
| Custom recovery / retry logic | **LangGraph** — add dedicated retry nodes |
| Beginner-friendly development | **Agno** — less framework-specific concepts to learn |

### Data Source Hierarchy (identical in both)
```
Query
 │
 ├─1──▶ search_taxonomy   → genre concept → curated cluster + film list
 ├─2──▶ search_abt         → 9,742 films, rich ABT metadata
 ├─3──▶ search_oscars_rag  → 2026 Oscar nominations (PDF chunks)
 ├─4──▶ search_tmdb_by_id  → live API, full cast/crew/budget details
 └─5──▶ search_web         → streaming, reviews, breaking news
```
